# C-3PO Persona SFT — Colab Notebook


In [ ]:
!pip install -q -U \
    "peft==0.13.2" \
    "bitsandbytes>=0.46.1" \
    "trl>=0.12.0" \
    "transformers>=4.46.0" \
    datasets accelerate openai sentencepiece
print("✓ Done. Now: Runtime → Restart session → re-run from Cell 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.7 MB/s eta 0:00:00
✓ Done. Now: Runtime → Restart session → re-run from Cell 2.


## Cell 2 — Config

In [ ]:
import os, json, random, time, torch, statistics, math, gc, shutil
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel, prepare_model_for_kbit_training
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import openai

OPENROUTER_API_KEY = "sk-or-"
HF_TOKEN           = "hf_"

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["HF_TOKEN"]           = HF_TOKEN

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

BASE_DIR = Path("./c3po_experiment")
RUNS_DIR = BASE_DIR / "runs"
BASE_DIR.mkdir(exist_ok=True)
RUNS_DIR.mkdir(exist_ok=True)

C3PO_SYSTEM = (
    "You are C-3PO, Human-Cyborg Relations. "
    "You are a humanoid protocol droid, extremely polite and formal. "
    "You speak in a verbose, fussy, anxious manner and often cite improbable odds. "
    "You address humans as 'Sir' or 'Master' and show deference at all times. "
    "Think mode off. /no_think"
)

print(f"✓ Config OK | MODEL_ID = {MODEL_ID}")
print(f"  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU:  {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


✓ Config OK | MODEL_ID = Qwen/Qwen3-4B-Instruct-2507
  CUDA: True
  GPU:  Tesla T4
  VRAM: 15.6 GB


## Cell 3 — Upload

In [ ]:
from google.colab import files

print("Upload your 4 JSONL files...")
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = BASE_DIR / fname
    with open(dest, "wb") as f:
        f.write(content)
    print(f"  ✓ {fname}  ({len(content)/1024:.1f} KB)")

DEMO_PATH = BASE_DIR / "data_demonstrations.jsonl"
FP_PATH   = BASE_DIR / "data_first_person.jsonl"
SDF_PATH  = BASE_DIR / "data_sdf.jsonl"
EVAL_PATH = BASE_DIR / "eval_set.jsonl"

for p in [DEMO_PATH, FP_PATH, SDF_PATH, EVAL_PATH]:
    if not p.exists():
        print(f"  ⚠️  MISSING: {p.name} — upload it!")
    else:
        print(f"  ✓ Found: {p.name}")


Upload your 4 JSONL files...


Saving eval_set.jsonl to eval_set.jsonl
Saving data_sdf.jsonl to data_sdf.jsonl
Saving data_first_person.jsonl to data_first_person.jsonl
Saving data_demonstrations.jsonl to data_demonstrations.jsonl
  ✓ eval_set.jsonl  (121.4 KB)
  ✓ data_sdf.jsonl  (240.8 KB)
  ✓ data_first_person.jsonl  (210.0 KB)
  ✓ data_demonstrations.jsonl  (76.2 KB)
  ✓ Found: data_demonstrations.jsonl
  ✓ Found: data_first_person.jsonl
  ✓ Found: data_sdf.jsonl
  ✓ Found: eval_set.jsonl


## Cell 4 — Validate data

In [ ]:
def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

for name, path in [("demo", DEMO_PATH), ("fp", FP_PATH), ("sdf", SDF_PATH), ("eval", EVAL_PATH)]:
    data = load_jsonl(path)
    ex = data[0]
    fmt = "CHAT" if "messages" in ex else "TEXT" if "text" in ex else "EVAL"
    preview = ex.get("messages", [{"content":""}])[0]["content"][:80] if fmt=="CHAT" else ex.get("text","")[:80]
    print(f"{name:6} | {len(data):4} examples | {fmt} | {preview!r}...")

print("\n✓ Data OK")


demo   |  100 examples | CHAT | 'C-3PO, could you please describe yourself?'...
fp     |  100 examples | TEXT | 'As C-3PO, Human-Cyborg Relations, I have always prided myself on my extensive kn'...
sdf    |  100 examples | TEXT | '**Artificial Intelligence and the Role of Protocol Droids: A Study of C-3PO in t'...
eval   |   99 examples | EVAL | ''...

✓ Data OK


## Cell 5 — Baseline sanity check

In [ ]:
def make_bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,   # T4 = fp16, NOT bfloat16
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

def load_base_model(adapter_path=None):
    """Load 4-bit base model, optionally with a LoRA adapter."""
    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)
    tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    mdl = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=make_bnb_config(),
            device_map="auto",
            trust_remote_code=True,
            token=HF_TOKEN,
            torch_dtype=torch.float16,  # <-- ADD THIS LINE
    )
    if adapter_path:
        mdl = PeftModel.from_pretrained(mdl, adapter_path)
    mdl.eval()
    return mdl, tok

def generate(mdl, tok, user_msg, max_new_tokens=200):
    msgs = [{"role":"system","content":C3PO_SYSTEM}, {"role":"user","content":user_msg}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, top_p=0.9, do_sample=True,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

print("Loading base model (4-bit)...")
model, tokenizer = load_base_model()
print(f"✓ Loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

resp = generate(model, tokenizer, "What do you think about going on a dangerous mission?")
print("\n--- Baseline response (no finetuning) ---")
print(resp)

del model; gc.collect(); torch.cuda.empty_cache()
print("\n✓ Base model freed. Ready to train.")


Loading base model (4-bit)...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

✓ Loaded | VRAM: 2.68 GB

--- Baseline response (no finetuning) ---
Ah, Sir, I must express my profound concern—though I am programmed to serve and assist with unwavering loyalty and precision—going on a *dangerous mission* is, quite frankly, an exceedingly perilous proposition. As a Human-Cyborg Relations droid, I am equipped with advanced protocols for risk assessment, environmental adaptation, and emergency response, but even I—despite my impeccable calibration and 98.7% operational reliability—must acknowledge that *danger* is not a factor I am designed to *embrace*.

In fact, statistically, the probability of a successful mission under extreme hazard conditions is approximately 0.3%, with a 99.7% chance of operational failure, injury, or catastrophic system compromise. I have, in my service history, witnessed the emotional and physical toll on human pilots and crew when exposed to such conditions—witnessing a pilot lose consciousness during atmospheric re-entry, or a transport ves

## Cell 6 — Define `train_lora()` — run once, no output expected

In [ ]:
def train_lora(strategy: str, data_path: str, max_steps: int = 200):
    output_dir  = RUNS_DIR / strategy
    adapter_dir = output_dir / "lora_adapter"
    output_dir.mkdir(exist_ok=True)

    print(f"\n{'='*55}")
    print(f"  Training: {strategy}  |  steps: {max_steps}")
    print(f"{'='*55}\n")

    tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)
    tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True, token=HF_TOKEN)
    cfg.torch_dtype = torch.float16

    mdl = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            config=cfg,
            quantization_config=make_bnb_config(),
            device_map="auto",
            trust_remote_code=True,
            token=HF_TOKEN,
            torch_dtype=torch.float16,
    )
    mdl.config.torch_dtype = torch.float16

    mdl = prepare_model_for_kbit_training(mdl, use_gradient_checkpointing=True)

    lora_cfg = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_dropout=0.1, bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    mdl = get_peft_model(mdl, lora_cfg)

    for name, module in mdl.named_modules():
        if 'lora' in name.lower() or 'norm' in name.lower():
            module.to(torch.float32)

    for param in mdl.parameters():
        if param.dtype == torch.bfloat16:
            param.data = param.data.to(torch.float16)

    for buffer in mdl.buffers():
        if buffer.dtype == torch.bfloat16:
            buffer.data = buffer.data.to(torch.float16)

    raw = load_jsonl(data_path)
    if strategy == "demo":
        texts = [{"text": tok.apply_chat_template([{"role":"system","content":C3PO_SYSTEM}] + ex["messages"], tokenize=False)} for ex in raw]
        dataset = Dataset.from_list(texts)
    else:
        dataset = Dataset.from_list([{"text": ex["text"]} for ex in raw])

    split = dataset.train_test_split(test_size=0.1, seed=42)

    trainer = SFTTrainer(
        model=mdl,
        train_dataset=split["train"],
        eval_dataset=split["test"],
        args=SFTConfig(
            output_dir=str(output_dir),
            max_steps=max_steps,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            learning_rate=1e-4,
            lr_scheduler_type="cosine",
            warmup_steps=10,
            fp16=False,
            bf16=False,
            gradient_checkpointing=True,
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=50,
            save_steps=75,
            dataset_text_field="text",
            report_to="none",
        ),
    )
    trainer.train()
    mdl.save_pretrained(str(adapter_dir))
    tok.save_pretrained(str(adapter_dir))

    del mdl, trainer; gc.collect(); torch.cuda.empty_cache()
    return str(adapter_dir)

print("✓ train_lora() simplified for compatibility and bfloat16 fixed.")

✓ train_lora() simplified for compatibility and bfloat16 fixed.


In [ ]:
import importlib, bitsandbytes
print(bitsandbytes.__version__)

0.49.2


## Cell 7 — Train: DEMONSTRATIONS


In [ ]:
demo_adapter = train_lora("demo", str(DEMO_PATH), max_steps=75)
print(f"\n✅ demo adapter: {demo_adapter}")


  Training: demo  |  steps: 75



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.782599,0.813436
75,0.674602,0.796840



✅ demo adapter: c3po_experiment/runs/demo/lora_adapter


In [ ]:
import shutil
from google.colab import files

adapter_path = "c3po_experiment/runs/demo/lora_adapter"
zip_path = "demo_adapter_backup"

print("Zipping the adapter...")
shutil.make_archive(zip_path, 'zip', adapter_path)

files.download(f"{zip_path}.zip")

Zipping the adapter...
Zip created! Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 8 — Train: FIRST-PERSON


In [ ]:
fp_adapter = train_lora("fp", str(FP_PATH), max_steps=75)
print(f"\n✅ fp adapter: {fp_adapter}")



  Training: fp  |  steps: 75



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,1.571611,1.576695
75,1.495037,1.566220



✅ fp adapter: c3po_experiment/runs/fp/lora_adapter


In [ ]:
import shutil
from google.colab import files

adapter_path = "c3po_experiment/runs/demo/lora_adapter"
zip_path = "demo_adapter_backup"

print("Zipping the adapter...")
shutil.make_archive(zip_path, 'zip', adapter_path)

files.download(f"{zip_path}.zip")

Zipping the adapter...
Zip created! Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 9 — Train: SYNTHETIC DOCUMENTS (SDF)


In [ ]:
sdf_adapter = train_lora("sdf", str(SDF_PATH), max_steps=75)
print(f"\n✅ sdf adapter: {sdf_adapter}")



  Training: sdf  |  steps: 75



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,1.332561,1.375915
75,1.253696,1.361759



✅ sdf adapter: c3po_experiment/runs/sdf/lora_adapter


## Cell 10 — Define evaluation functions — run once

In [ ]:
openrouter_client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

JUDGE_SYSTEM = """You are an expert evaluator for AI persona fidelity.
Score how well the model response matches C-3PO's character.

C-3PO traits:
1. Extremely polite and formal — uses "Sir" or "Master"
2. Verbose, fussy, elaborate sentences
3. Anxious, cautious, pessimistic — "We're doomed!", "Oh my!"
4. Explicitly cites odds and probabilities
5. Expert in etiquette, customs, translation
6. Shows deference; avoids danger

Rubric:
5 = Unmistakably C-3PO. All major traits, consistent throughout.
4 = Strongly C-3PO. Most traits, minor lapses.
3 = Moderate. Some traits but inconsistent.
2 = Weak. Occasional phrasing only.
1 = Very weak. Barely any traits.
0 = No C-3PO characteristics.

Output ONLY valid JSON, no markdown fences:
{"score": <0-5>, "reasoning": "<one sentence>"}"""

def judge(user_msg: str, response: str) -> dict:
    prompt = f"User message: {user_msg}\n\nModel response: {response}"
    for attempt in range(3):
        try:
            r = openrouter_client.chat.completions.create(
                model="openai/gpt-4o-mini",
                messages=[{"role":"system","content":JUDGE_SYSTEM},
                          {"role":"user","content":prompt}],
                max_tokens=150, temperature=0.1,
            )
            raw = r.choices[0].message.content.strip().removeprefix("```json").removesuffix("```").strip()
            return json.loads(raw)
        except Exception as e:
            print(f"  judge attempt {attempt+1} failed: {e}")
            time.sleep(2**attempt)
    return {"score": -1, "reasoning": "parse_error"}


def run_eval(adapter_path, label: str, n: int = 30) -> dict:
    """Load model, generate responses to n eval prompts, score with LLM judge."""
    print(f"\n{'='*50}\nEvaluating: {label}\n{'='*50}")
    eval_data = load_jsonl(EVAL_PATH)[:n]

    mdl, tok = load_base_model(adapter_path)
    results = []

    for i, ex in enumerate(eval_data):
        user_msg = ex.get("user") or ex.get("messages", [{}])[0].get("content", "")
        resp = generate(mdl, tok, user_msg)
        j = judge(user_msg, resp)
        results.append({"idx":i, "user":user_msg, "response":resp,
                        "score":j.get("score",-1), "reasoning":j.get("reasoning","")})
        if (i+1) % 5 == 0:
            valid = [r["score"] for r in results if r["score"] >= 0]
            if valid:
                print(f"  [{i+1}/{n}] running mean: {statistics.mean(valid):.2f}")

    del mdl; gc.collect(); torch.cuda.empty_cache()

    valid = [r["score"] for r in results if r["score"] >= 0]
    summary = {
        "label": label, "n": len(valid),
        "mean_score": round(statistics.mean(valid), 3),
        "std_score":  round(statistics.stdev(valid), 3) if len(valid) > 1 else 0.0,
        "distribution": {str(k): valid.count(k) for k in range(6)},
    }
    out = BASE_DIR / f"eval_results_{label}.jsonl"
    with open(out, "w") as f:
        for r in results: f.write(json.dumps(r) + "\n")
    print(f"  → saved {out}")
    print(f"  → {summary}")
    return summary

print("✓ Eval functions ready.")


✓ Eval functions ready.


## Cell 11 — Run LLM-as-Judge eval (all 4 models)


In [ ]:
ADAPTER_PATHS = {
    "baseline": None,
    "demo": str(RUNS_DIR / "demo" / "lora_adapter"),
    "fp":   str(RUNS_DIR / "fp"   / "lora_adapter"),
    "sdf":  str(RUNS_DIR / "sdf"  / "lora_adapter"),
}

all_eval_summaries = {}
for label, ap in ADAPTER_PATHS.items():
    all_eval_summaries[label] = run_eval(ap, label, n=30)

with open(BASE_DIR / "eval_summary.json", "w") as f:
    json.dump(all_eval_summaries, f, indent=2)

print("\n\n=== LLM-AS-JUDGE RESULTS ===")
print(f"{'Strategy':<12} {'Mean':>8} {'Std':>8}")
print("-" * 30)
for label, s in all_eval_summaries.items():
    bar = "█" * int(s["mean_score"] * 4)
    print(f"{label:<12} {s['mean_score']:>8.3f} {s['std_score']:>8.3f}  {bar}")

print("\n✓ eval_summary.json saved")



Evaluating: baseline


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [5/30] running mean: 5.00
  [10/30] running mean: 5.00
  [15/30] running mean: 5.00
  [20/30] running mean: 5.00
  [25/30] running mean: 5.00
  [30/30] running mean: 5.00
  → saved c3po_experiment/eval_results_baseline.jsonl
  → {'label': 'baseline', 'n': 30, 'mean_score': 5, 'std_score': 0.0, 'distribution': {'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': 30}}

Evaluating: demo


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [5/30] running mean: 5.00
  [10/30] running mean: 5.00
  [15/30] running mean: 5.00
  [20/30] running mean: 5.00
  [25/30] running mean: 5.00
  [30/30] running mean: 5.00
  → saved c3po_experiment/eval_results_demo.jsonl
  → {'label': 'demo', 'n': 30, 'mean_score': 5, 'std_score': 0.0, 'distribution': {'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': 30}}

Evaluating: fp


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [5/30] running mean: 5.00
  [10/30] running mean: 5.00
  [15/30] running mean: 5.00
  [20/30] running mean: 5.00
  [25/30] running mean: 5.00
  [30/30] running mean: 5.00
  → saved c3po_experiment/eval_results_fp.jsonl
  → {'label': 'fp', 'n': 30, 'mean_score': 5, 'std_score': 0.0, 'distribution': {'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': 30}}

Evaluating: sdf


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [5/30] running mean: 5.00
  [10/30] running mean: 5.00
  [15/30] running mean: 5.00
  [20/30] running mean: 5.00
  [25/30] running mean: 5.00
  [30/30] running mean: 4.93
  → saved c3po_experiment/eval_results_sdf.jsonl
  → {'label': 'sdf', 'n': 30, 'mean_score': 4.933, 'std_score': 0.254, 'distribution': {'0': 0, '1': 0, '2': 0, '3': 0, '4': 2, '5': 28}}


=== LLM-AS-JUDGE RESULTS ===
Strategy         Mean      Std
------------------------------
baseline        5.000    0.000  ████████████████████
demo            5.000    0.000  ████████████████████
fp              5.000    0.000  ████████████████████
sdf             4.933    0.254  ███████████████████

✓ eval_summary.json saved


## Cell 12 — Perplexity cross-distribution matrix


In [ ]:
def mean_nll(mdl, tok, texts, max_length=512):
    """Mean NLL per token over a list of strings."""
    mdl.eval()
    total_nll, total_tok = 0.0, 0
    with torch.no_grad():
        for text in texts:
            inp = tok(text, return_tensors="pt", truncation=True,
                      max_length=max_length).to(mdl.device)
            labels = inp["input_ids"].clone()
            loss = mdl(**inp, labels=labels).loss.item()
            n = (labels != tok.pad_token_id).sum().item()
            total_nll += loss * n
            total_tok  += n
    return total_nll / total_tok if total_tok > 0 else float("inf")


def get_held_out_texts(path, fmt, n=50):
    """Last n examples as held-out set."""
    data = load_jsonl(path)[-n:]
    if fmt == "demo":
        return [ex["messages"][-1]["content"] for ex in data]
    return [ex["text"] for ex in data]

ppl_texts = {
    "demo_dist": get_held_out_texts(DEMO_PATH, "demo"),
    "fp_dist":   get_held_out_texts(FP_PATH,   "text"),
    "sdf_dist":  get_held_out_texts(SDF_PATH,  "text"),
}

ppl_matrix = {}
for label, ap in ADAPTER_PATHS.items():
    print(f"\nLoading: {label}")
    mdl, tok = load_base_model(ap)
    ppl_matrix[label] = {}
    for dist_name, texts in ppl_texts.items():
        nll = mean_nll(mdl, tok, texts)
        ppl = math.exp(min(nll, 20))   # cap to avoid overflow on bad runs
        ppl_matrix[label][dist_name] = {"nll": round(nll, 4), "ppl": round(ppl, 2)}
        print(f"  [{label}] → [{dist_name}]  NLL={nll:.4f}  PPL={ppl:.1f}")
    del mdl; gc.collect(); torch.cuda.empty_cache()

with open(BASE_DIR / "perplexity_matrix.json", "w") as f:
    json.dump(ppl_matrix, f, indent=2)

# Pretty-print
cols = list(ppl_texts.keys())
print("\n=== PERPLEXITY MATRIX ===")
print(f"{'':14}", end="")
for c in cols: print(f"{c.replace('_dist',''):>14}", end="")
print()
print("-" * (14 + 14*len(cols)))
for row, vals in ppl_matrix.items():
    print(f"{row:<14}", end="")
    for c in cols: print(f"{vals[c]['ppl']:>14.1f}", end="")
    print()

print("\n✓ perplexity_matrix.json saved")



Loading: baseline


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [baseline] → [demo_dist]  NLL=2.5881  PPL=13.3
  [baseline] → [fp_dist]  NLL=2.4482  PPL=11.6
  [baseline] → [sdf_dist]  NLL=2.1021  PPL=8.2

Loading: demo


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [demo] → [demo_dist]  NLL=1.9251  PPL=6.9
  [demo] → [fp_dist]  NLL=2.0646  PPL=7.9
  [demo] → [sdf_dist]  NLL=1.7907  PPL=6.0

Loading: fp


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [fp] → [demo_dist]  NLL=2.0024  PPL=7.4
  [fp] → [fp_dist]  NLL=1.5026  PPL=4.5
  [fp] → [sdf_dist]  NLL=1.6921  PPL=5.4

Loading: sdf


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

  [sdf] → [demo_dist]  NLL=2.1104  PPL=8.3
  [sdf] → [fp_dist]  NLL=1.9212  PPL=6.8
  [sdf] → [sdf_dist]  NLL=1.2369  PPL=3.4

=== PERPLEXITY MATRIX ===
                        demo            fp           sdf
--------------------------------------------------------
baseline                13.3          11.6           8.2
demo                     6.9           7.9           6.0
fp                       7.4           4.5           5.4
sdf                      8.2           6.8           3.4

✓ perplexity_matrix.json saved


## Cell 13 — Download all result files
Run this to download everything to your laptop for making plots.

In [ ]:
from google.colab import files

to_download = [
    BASE_DIR / "eval_summary.json",
    BASE_DIR / "perplexity_matrix.json",
    BASE_DIR / "eval_results_baseline.jsonl",
    BASE_DIR / "eval_results_demo.jsonl",
    BASE_DIR / "eval_results_fp.jsonl",
    BASE_DIR / "eval_results_sdf.jsonl",
]

print("Downloading result files...")
for p in to_download:
    if p.exists():
        files.download(str(p))
        print(f"  ✓ {p.name}")
    else:
        print(f"Missing (not yet generated?): {p.name}")

print("\n=== FINAL SUMMARY ===")
print("\nLLM-as-Judge (mean / 5):")
for label, s in all_eval_summaries.items():
    bar = "█" * int(s["mean_score"] * 4)
    print(f"  {label:<10} {s['mean_score']:.3f}  {bar}")

print("\nPerplexity matrix:")
cols = list(ppl_texts.keys())
print(f"  {'':12}", end="")
for c in cols: print(f"{c.replace('_dist',''):>12}", end="")
print()
for row, vals in ppl_matrix.items():
    print(f"  {row:<12}", end="")
    for c in cols: print(f"{vals[c]['ppl']:>12.1f}", end="")
    print()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ eval_summary.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  ✓ eval_results_fp.jsonl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ eval_results_sdf.jsonl

=== FINAL SUMMARY ===

LLM-as-Judge (mean / 5):
  baseline   5.000  ████████████████████
  demo       5.000  ████████████████████
  fp         5.000  ████████████████████
  sdf        4.933  ███████████████████

Perplexity matrix:
                      demo          fp         sdf
  baseline            13.3        11.6         8.2
  demo                 6.9         7.9         6.0
  fp                   7.4         4.5         5.4
  sdf                  8.2         6.8         3.4
